In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Schreibe dein erstes Qiskit Serverless-Programm

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</details>
Dieses Beispiel zeigt, wie du `qiskit-serverless`-Tools verwendest, um ein paralleles Transpilierungsprogramm zu erstellen, und anschließend `qiskit-ibm-catalog` einsetzt, um dein Programm auf der IBM Quantum Platform als wiederverwendbaren Remote-Dienst bereitzustellen.
## Beispiel: Remote-Transpilierung mit Qiskit Serverless
Beginne mit dem folgenden Beispiel, das einen `circuit` für ein gegebenes `backend` und einen Ziel-`optimization_level` transpiliert, und füge schrittweise weitere Elemente hinzu, um deine Arbeitslast in Qiskit Serverless bereitzustellen.

Füge den folgenden Code in die Datei `./source_files/transpile_remote.py` ein. Diese Datei ist das Programm, das du in Qiskit Serverless hochlädst.

In [1]:
# This cell is hidden from users, it just creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [2]:
%%writefile ./source_files/transpile_remote.py

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

Writing ./source_files/transpile_remote.py


## Set up your files

Qiskit Serverless requires setting up your workload’s `.py` files into a dedicated directory. The following structure is an example of good practice:

```text
serverless_program
├── program_uploader.ipynb
└── source_files
    ├── transpile_remote.py
    └── *.py
```

Serverless uploads the contents of `source_files` to run remotely. Once these are set up, you can adjust `transpile_remote.py` to fetch inputs and return outputs.

### Get program arguments

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [3]:
%%writefile --append ./source_files/transpile_remote.py

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

Appending to ./source_files/transpile_remote.py


## Dateien einrichten
Qiskit Serverless erfordert, dass du die `.py`-Dateien deiner Arbeitslast in einem eigenen Verzeichnis organisierst. Die folgende Struktur ist ein Beispiel für gute Praxis:

In [4]:
%%writefile --append ./source_files/transpile_remote.py

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

Appending to ./source_files/transpile_remote.py


Serverless lädt den Inhalt von `source_files` hoch, um ihn remote auszuführen. Sobald diese Dateien eingerichtet sind, kannst du `transpile_remote.py` anpassen, um Eingaben entgegenzunehmen und Ausgaben zurückzugeben.

### Programmargumente abrufen
Deine initiale `transpile_remote.py` hat drei Eingaben: `circuits`, `backend_name` und `optimization_level`. Serverless ist derzeit darauf beschränkt, nur serialisierbare Eingaben und Ausgaben zu akzeptieren. Aus diesem Grund kannst du `backend` nicht direkt übergeben — verwende stattdessen `backend_name` als String.

In [5]:
%%writefile --append ./source_files/transpile_remote.py

results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

Appending to ./source_files/transpile_remote.py


## Deploy to IBM Quantum Platform

The previous section created a program to be run remotely. The code cells in this section upload that program to Qiskit Serverless.

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key, which you can find on the [IBM Quantum dashboard](https://quantum.cloud.ibm.com), and upload the program.

You can use `save_account()` to save your credentials (See the [Set up to use IBM Cloud](/docs/guides/cloud-setup#cloud-save) section). Note that this writes your credentials to the same file as [`QiskitRuntimeService.save_account()`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account).

In [6]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

An diesem Punkt kannst du dein Backend mit `QiskitRuntimeService` abrufen und dein vorhandenes Programm mit dem folgenden Code ergänzen. Der folgende Code setzt voraus, dass du deine Zugangsdaten bereits [gespeichert hast](/guides/cloud-setup).

In [7]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [8]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

Schließlich kannst du `transpile_remote()` für alle übergebenen `circuits` ausführen und die `transpiled_circuits` als Ergebnis zurückgeben:

In [9]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [10]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [11]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

Qiskit Serverless komprimiert den Inhalt von `working_dir` (in diesem Fall `source_files`) in ein `tar`-Archiv, das hochgeladen und danach bereinigt wird. Der `entrypoint` identifiziert die ausführbare Hauptprogrammdatei, die Qiskit Serverless ausführen soll. Falls dein Programm benutzerdefinierte `pip`-Abhängigkeiten hat, kannst du diese in einem `dependencies`-Array angeben: